# Data Cleaning and Dataset Splitting

This notebook performs the initial data preparation steps for the credit card fraud detection project.
The goal of this stage is to clean the raw dataset, validate its integrity, and prepare the data for machine learning experiments by creating structured training and testing datasets.

## Objectives

The main goals of this notebook are:

- Clean and validate the raw dataset
- Prepare the data for machine learning experiments
- Separate features and target variables
- Split the dataset into training and testing sets

## Configuration

All project configurations are centralized in the configuration.yaml file located in the project root directory.
This configuration file defines parameters related to:

- Data paths
- Dataset split configuration
- Experiment settings

Loading configuration values from a single source ensures consistency and reproducibility across notebooks and pipeline components.

## Data Source

The raw dataset used in this notebook is located at:

    ../data/raw/creditcard.csv

This path is relative to the notebook location within the project structure, since the notebook and dataset are stored in different folders inside the project root.

## Processing Steps

The following operations are performed in this notebook:

1. Load the raw dataset
2. Validate dataset integrity and inspect basic statistics
3. Perform data cleaning operations
4. Prepare the feature matrix (X) and target variable (y)
5. Split the dataset into training and testing sets

## Generated Datasets

The processed datasets generated in this step are saved to the following locations:

    ../data/cleaned/creditcard_cleaned.parquet
    ../data/splits/train.parquet
    ../data/splits/test.parquet

These paths are relative to the notebook location within the project structure.
The generated datasets are used as inputs for the subsequent stages of the machine learning workflow.

## Research vs Production Code

This notebook was created during the research and experimentation phase of the project to prototype the data preparation workflow.
The final production-ready implementation of the data preparation pipeline is available in:

    src/datapipeline

This separation ensures that the production pipeline remains modular, maintainable, and reproducible, while notebooks are used primarily for experimentation and analysis.

In [ ]:
import pandas as pd
from pathlib import Path
from typing import Tuple
from sklearn.model_selection import train_test_split
import yaml

# Configurations

In [ ]:
#There is a config.yaml file at the project root
#loading config file
config_path = '../config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
target_column = config['parameters']['target_column']
raw_dataset_path = config['datasets']['raw_dataset_path']  
cleaned_df_path = config['datasets']['cleaned_dataset_path'] # location to save the cleaned dataset
train_dataset_path = config['datasets']['train_dataset_path']
test_dataset_path = config['datasets']['test_dataset_path']
test_size = config['2-data_cleaning_and_split']['test_size'] # proportion of the dataset to include in the test split
random_state = config['parameters']['random_state'] # Controls the shuffling applied to the data before applying the split
min_samples = config['2-data_cleaning_and_split']['min_samples'] # minimun number of samples the dataset must contain
num_classes = config['2-data_cleaning_and_split']['num_classes'] # minimum number of target classes the dataset must contain


# Load Data

In [ ]:
dataset_path = Path(raw_dataset_path).resolve()
dataset_path

In [ ]:
data = pd.read_csv(dataset_path)

In [ ]:
data.describe()

# Validate Data

In [ ]:
def validate_data(
        df: pd.DataFrame,
        target_column: str,
        min_samples: int = 1000,
        num_classes: int = 2,
) -> None:
    
    """
    Validates if the dataset is valid

    Args:
        df (pd.DataFrame): pandas dataframe after preliminary cleaning
        target_column (str): target column
        min_samples (int, optional): required min number of samples. Defaults to 1000.
        num_classes (int, optional): number of classes in the target. Defaults to 2.
        logger (logging.Logger, optional): Logger instance.
  
    Raises:
        ValueError: the number of samples is less than min_samples
        ValueError: the number of classes is not equal to num_classes
        ValueError: one target class has no samples
    """

    print('Validating Data')
    
    if df.shape[0] < min_samples:
        raise ValueError(f"Dataset must have at least {min_samples} samples")
    
    if target_column not in df.columns:
        raise ValueError(f"Target column '{target_column}' not found")

    class_count = df[target_column].value_counts(dropna=True)

    if len(class_count) != num_classes:
        raise ValueError(f"Expected {num_classes} classes, found {len(class_count)}")
    
    if class_count.min() <= 0:
     raise ValueError("One target class has no samples")
    print('Dataset Validated')

In [ ]:
validate_data(
        df = data,
        target_column = target_column,
        min_samples = min_samples,
        num_classes =num_classes)


# Data Cleaning

In [ ]:
# Path to save the cleaned dataframe
#verifying if the folder existis
parent_cleaned_df_path = Path(cleaned_df_path).parent.resolve()
parent_cleaned_df_path.mkdir(parents=True, exist_ok=True)


In [ ]:
def clean_data(df: pd.DataFrame, 
               target_column: str, 
) -> Tuple[pd.DataFrame, int]:

    """
    Perform preliminary data cleaning.

    Steps:
    - Remove duplicate rows
    - Remove rows without target
    - Sanity check on target values

    Args:
        df (pd.DataFrame): Input dataframe.
        target_column (str): Name of target column.
        logger (logging.Logger, optional): Logger instance.

    Returns:
        Tuple[pd.DataFrame, int]: Cleaned dataframe and number of rows removed.
    """

    initial_rows = df.shape[0]

    # Remove duplicates
    df = df.drop_duplicates()
    duplicated_rows_removed = initial_rows - df.shape[0]
    
    print(f'Removed {duplicated_rows_removed} duplicated rows')


    # Remove rows without target
    before = df.shape[0]
    df = df.dropna(subset=[target_column])
    removed_missing_target = before - df.shape[0]
    
    print(f'Removed {removed_missing_target} rows without target')

    # Sanity checks
    if (df[target_column] < 0).any():
        raise ValueError("Invalid target values detected")

    total_removed = duplicated_rows_removed + removed_missing_target
    print(f'Total rows removed: {total_removed}')

    return df, total_removed



In [ ]:
df_cleaned, _ = clean_data(data,
                       target_column)


In [ ]:
df_cleaned.to_parquet(cleaned_df_path)

# Data Splitting

In [ ]:
#Path to save the training and testing dataframes
splitted_datasets_path = Path(train_dataset_path).parent.resolve()
splitted_datasets_path.mkdir(parents=True, exist_ok=True)


In [ ]:
def split_data(df: pd.DataFrame,
               target_column: str,
               test_size: float,
               random_state: int,
               ) -> Tuple[pd.DataFrame, pd.DataFrame]:

    """
    Split a DataFrame into training and testing sets.

    Args:
        df (pd.DataFrame): The DataFrame to split.
        target_column (str): The name of the target column.
        test_size (float): The proportion of the dataset to include in the test split.
        random_state (int): The seed used by the random number generator.
        logger (logging.Logger | None, optional): The logger to use. Defaults to None.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: A tuple containing the training and testing DataFrames.
    """

    X = df.drop(target_column, axis=1)
    y = df[target_column]

    print('Spliting data...')

    if y.nunique() < 2:
        raise ValueError("Target column must have at least two classes for stratified split")

        
    X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                        test_size=test_size, 
                                                        random_state=random_state,
                                                        stratify=y)
    print(f'{df.shape[0]} samples split into {X_train.shape[0]} ' 
                    f'train samples and {X_test.shape[0]} test samples')
    print(f'{X_train.shape[0]/df.shape[0]*100:.2f}% of the '
                    f'dataset is used for training and {X_test.shape[0]/df.shape[0]*100:.2f}% for testing' )

    df_train = X_train
    df_test  = X_test
    df_train[target_column] = y_train
    df_test[target_column] = y_test

    return df_train, df_test

In [ ]:
X_train, X_test = split_data(df = df_cleaned, 
                            target_column = target_column,
                            test_size = test_size,
                            random_state = random_state)

In [ ]:
X_train.to_parquet(train_dataset_path)
X_test.to_parquet(test_dataset_path)